<a href="https://colab.research.google.com/github/ajayrfhp/LearningDeepLearning/blob/main/self_attention_yet_again.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import torch
import torch.nn as nn

In [17]:
def run_self_attention(X, d_head, W_Q, W_K, W_V):

  Q = X @ W_Q # shape of Q is (batch_size, seq_length, d_head)
  K = X @ W_K # shape of K is (batch_size, seq_length, d_head)
  V = X @ W_V # shape of V is (batch_size, seq_length, d_head)

  KT = K.transpose(1, 2)
  self_attn_weights =  nn.Softmax(dim=-1)((Q @ KT)/ torch.sqrt(torch.tensor(d_head))) # (batch_size, seq_length, seq_length)

  outputs = self_attn_weights @ V # (batch_size, seq_length, d_head)
  return outputs, self_attn_weights

In [35]:
batch_size = 1
seq_length = 10
d_model = 32
d_head = 64

X = torch.randn((batch_size, seq_length, d_model))
W_Q = torch.randn((d_model, d_head))
W_V = torch.randn((d_model, d_head))
W_K = torch.randn((d_model, d_head))

# self attention will transform this into ((batch_size, seq_length, d_head))



outputs, self_attn_weights = run_self_attention(X, d_head, W_Q, W_K, W_V)

outputs.shape

torch.Size([1, 10, 64])

In [19]:
# rows should normalize to 1 in self_attention_weights

assert torch.allclose(self_attn_weights.sum(dim=-1), torch.ones((batch_size, seq_length)))

In [ ]:
self_attn_weights.sum(dim=-1).shape

## Equivariance test

In [34]:
# swap token 4 and 7
X_swapped = torch.clone(X).detach()
X_swapped[:,4,:] = X[:,7,:]
X_swapped[:,7,] = X[:,4,:]

swapped_outputs, swapped_self_attn_weights = run_self_attention(X_swapped, d_head, W_Q, W_K, W_V)

assert torch.allclose(swapped_outputs[:,4], outputs[:,7])
assert torch.allclose(swapped_outputs[:,7], outputs[:,4])

In [52]:
def run_mha_naive(X, d_model, d_head, n_heads):
  outputs_mha = []
  for i in range(n_heads):
    W_Q = torch.randn((d_model, d_head))
    W_V = torch.randn((d_model, d_head))
    W_K = torch.randn((d_model, d_head))
    outputs_sha, self_attn_weights = run_self_attention(X, d_head, W_Q, W_K, W_V)
    outputs_mha.append(outputs_sha.unsqueeze(dim=-1))
  W_O = torch.randn((d_head * n_heads, d_model))
  outputs_mha = torch.cat(outputs_mha, dim=-1) # (Batch_size, seq_length, d_head, n_heads)

  outputs_mha = outputs_mha.reshape((-1, X.shape[1] , d_head * n_heads))
  outputs = outputs_mha @ W_O # (Batch_size, seq_length, d_head * n_heads) @ (d_head * n_heads, d_model) = (Batch_size, seq_length, d_model)

  return outputs

n_heads=8
outputs_mha = run_mha_naive(X, d_model, d_head, n_heads)
outputs_mha.shape

torch.Size([1, 10, 32])

In [44]:
X.shape

torch.Size([1, 10, 32])

In [68]:
def run_mha(X, d_model, n_heads):
  d_head = d_model // n_heads
  sqrt_dk = torch.sqrt(torch.tensor(d_head, dtype=torch.float32))

  W_Q = torch.randn((d_model, d_model))
  W_V = torch.randn((d_model, d_model))
  W_K = torch.randn((d_model, d_model))
  W_O = torch.randn((d_model, d_model))

  # shape of X is ((batch_size, seq_length, d_model))
  Q = X @ W_Q
  K = X @ W_K
  V = X @ W_V

  # shape of Q is (batch_size, seq_length, d_model)

  Q = (Q.reshape((-1, X.shape[1], n_heads, d_head))).permute(0, 2, 1, 3)
  KT = (K.reshape((-1, X.shape[1], n_heads, d_head))).permute(0, 2, 3, 1)
  V = (V.reshape((-1, X.shape[1], n_heads, d_head))).permute(0, 2, 1, 3)



  # shape of KT is (batch_size, n_heads, d_head, seq_length)
  raw_sna = (Q @ KT) /sqrt_dk
  # (batch_size, n_heads, seq_length, d_head) @ (batch_size, n_heads, d_head, seq_length) = (batch_size, n_heads, seq_length, seq_length)

  sna = nn.Softmax(dim=-1)(raw_sna)


  mha_outputs = sna @ V
  # (batch_size, n_heads, seq_length, seq_length) @ (batch_size, n_heads, seq_length, d_head) = (batch_size, n_heads, seq_length, d_head)

  mha_outputs = mha_outputs.permute(0, 2, 1, 3).reshape((-1, X.shape[1], d_head * n_heads))
  return mha_outputs @ W_O

mha_outputs = run_mha(X, d_model = 32, n_heads = 4)
mha_outputs.shape






torch.Size([1, 10, 32])

In [66]:

# 1D memory array representing d_model = 6
Q_flat = torch.tensor([10, 11, 12, 20, 21, 22])
n_heads, d_head = 2, 3

# Reshape: (n_heads=2, d_head=3)
Q_correct = Q_flat.reshape(n_heads, d_head)
Q_correct

tensor([[10, 11, 12],
        [20, 21, 22]])

In [67]:
Q_wrong = Q_flat.reshape((d_head, n_heads)).permute(1, 0)
Q_wrong

tensor([[10, 12, 21],
        [11, 20, 22]])